In [ ]:
!nvidia-smi

Mon Nov 23 03:16:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# paths
ZIP_PATH = "/content/drive/MyDrive/MOPG-7.zip"
WORK_DIR = "/content"

# copy & unzip
!cp "$ZIP_PATH" $WORK_DIR/data.zip
!unzip -o $WORK_DIR/data.zip -d $WORK_DIR

Archive:  /content/data.zip
   creating: /content/Dataset/
  inflating: /content/Dataset/classes.txt  
   creating: /content/Dataset/images/
  inflating: /content/Dataset/images/00001.jpg  
  inflating: /content/Dataset/images/000010.jpg  
  inflating: /content/Dataset/images/0000100.jpg  
  inflating: /content/Dataset/images/00001000.jpg  
  inflating: /content/Dataset/images/00001001.jpg  
  inflating: /content/Dataset/images/00001002.jpg  
  inflating: /content/Dataset/images/00001003.jpg  
  inflating: /content/Dataset/images/00001004.jpg  
  inflating: /content/Dataset/images/00001005.jpg  
  inflating: /content/Dataset/images/00001006.jpg  
  inflating: /content/Dataset/images/00001007.jpg  
  inflating: /content/Dataset/images/00001008.jpg  
  inflating: /content/Dataset/images/00001009.jpg  
  inflating: /content/Dataset/images/0000101.jpg  
  inflating: /content/Dataset/images/00001010.jpg  
  inflating: /content/Dataset/images/00001012.jpg  
  inflating: /content/Dataset/imag

In [ ]:
import glob
import os
import random
import shutil
from collections import defaultdict

random.seed(42)

SRC_ROOT = "/content/Dataset"
SPLIT_ROOT = "/content/Data_split"

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

# 1. Identify the dental center from the filename
def identify_center(img_path):
    filename = os.path.splitext(os.path.basename(img_path))[0].lower()

    if filename.startswith("0000"):
        return "Sonia"
    elif filename.startswith("ibn"):
        return "Ibn_Sina"
    elif filename.startswith("nir"):
        return "Niramoy"
    elif filename.startswith("h"):
        return "Health_City"
    else:
        return None


# 2. Find all images
all_images = [
    p for p in glob.glob(
        os.path.join(SRC_ROOT, "**", "*"),
        recursive=True
    )
    if p.lower().endswith(IMG_EXTS)
]

print("Total images found:", len(all_images))


# 3. Group images by dental center
center_groups = defaultdict(list)
unidentified = []

for img in all_images:
    center = identify_center(img)

    if center:
        center_groups[center].append(img)
    else:
        unidentified.append(img)

if unidentified:
    raise ValueError(
        f"{len(unidentified)} images have unidentified center prefixes. "
        f"Examples: {unidentified[:5]}"
    )

expected_centers = {
    "Sonia", "Ibn_Sina", "Niramoy", "Health_City"
}

if set(center_groups.keys()) != expected_centers:
    raise ValueError("One or more expected centers are missing.")

print("\nCenter-wise image counts:")
for center, images in center_groups.items():
    print(center, ":", len(images))


# 4. Find matching YOLO labels
def find_label(img_path):
    stem = os.path.splitext(os.path.basename(img_path))[0]

    candidates = glob.glob(
        os.path.join(SRC_ROOT, "**", stem + ".txt"),
        recursive=True
    )

    labels_only = [
        c for c in candidates
        if os.sep + "labels" + os.sep in c
    ]

    return (labels_only or candidates or [None])[0]


# 5. Split each center independently
splits = {
    "train": [],
    "val": [],
    "test": []
}

for center, images in center_groups.items():

    random.shuffle(images)

    n = len(images)

    n_train = int(n * 0.80)
    n_val = int(n * 0.10)

    train_images = images[:n_train]
    val_images = images[n_train:n_train + n_val]
    test_images = images[n_train + n_val:]

    splits["train"].extend(train_images)
    splits["val"].extend(val_images)
    splits["test"].extend(test_images)

    print(f"\n{center}:")
    print("Train:", len(train_images))
    print("Validation:", len(val_images))
    print("Test:", len(test_images))


# 6. Copy images and YOLO labels
for split_name, images in splits.items():

    img_dir = os.path.join(
        SPLIT_ROOT, "images", split_name
    )

    lbl_dir = os.path.join(
        SPLIT_ROOT, "labels", split_name
    )

    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for img_path in images:

        filename = os.path.basename(img_path)

        shutil.copy2(
            img_path,
            os.path.join(img_dir, filename)
        )

        lbl_path = find_label(img_path)

        if lbl_path:
            shutil.copy2(
                lbl_path,
                os.path.join(lbl_dir, os.path.basename(lbl_path))
            )


# 7. Final summary
print("\n========== FINAL DATA SPLIT ==========")

for split_name, images in splits.items():
    print(f"{split_name}: {len(images)} images")

print("Total:", sum(len(v) for v in splits.values()))

print("\nCenter-wise distribution in each split:")

for split_name, images in splits.items():
    print(f"\n{split_name.upper()}")

    for center in sorted(expected_centers):
        count = sum(
            1 for img in images
            if identify_center(img) == center
        )

        print(f"{center}: {count}")

print("\nCenter-wise splitting completed!")

Total images found: 1941

Center-wise image counts:
Ibn_Sina : 486
Sonia : 1338
Niramoy : 104
Health_City : 13

Ibn_Sina:
Train: 388
Validation: 48
Test: 50

Sonia:
Train: 1070
Validation: 133
Test: 135

Niramoy:
Train: 83
Validation: 10
Test: 11

Health_City:
Train: 10
Validation: 1
Test: 2

========== FINAL DATA SPLIT ==========
train: 1551 images
val: 192 images
test: 198 images
Total: 1941

Center-wise distribution in each split:

TRAIN
Health_City: 10
Ibn_Sina: 388
Niramoy: 83
Sonia: 1070

VAL
Health_City: 1
Ibn_Sina: 48
Niramoy: 10
Sonia: 133

TEST
Health_City: 2
Ibn_Sina: 50
Niramoy: 11
Sonia: 135

Center-wise splitting completed!


In [ ]:
classes_path = "/content/Dataset/classes.txt"
yaml_path = "/content/data.yaml"

with open(classes_path) as f:
    classes = [c.strip() for c in f if c.strip()]

with open(yaml_path, "w") as f:
    f.write(f"path: {SPLIT_ROOT}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("names:\n")
    for i, c in enumerate(classes):
        f.write(f"  {i}: {c}\n")

print("Classes:", classes)
print(open(yaml_path).read())


Classes: ['Missing', 'Dental Crown', 'Root Canal', 'Caries', 'Broken Down', 'Wisdom Teeth', 'No Target Findings']
path: /content/Data_split
train: images/train
val: images/val
names:
  0: Missing
  1: Dental Crown
  2: Root Canal
  3: Caries
  4: Broken Down
  5: Wisdom Teeth
  6: No Target Findings



In [ ]:
model_v8 = YOLO("yolov8m.pt")  # auto-downloads official YOLOv8m weights
model_v8.model.info()

YOLOv8m summary: 169 layers, 25,902,640 parameters, 0 gradients, 79.3 GFLOPs


(169, 25902640, 0, 79.313664)

In [ ]:
results_v8 = model_v8.train(
    model="yolo8m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov8"
)


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov8, nbs=64, nms=None, opset=None, optimize=False, optimizer=aut

In [ ]:
metrics_val_v8 = model_v8.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv8m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v8.box.map50)
print("mAP@0.5:0.95:", metrics_val_v8.box.map)
print("Precision   :", metrics_val_v8.box.mp)
print("Recall      :", metrics_val_v8.box.mr)


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,843,813 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3902.3±883.3 MB/s, size: 860.0 KB)
val: Scanning /content/Data_split/labels/val.cache... 192 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 192/192 47.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 2.1it/s 5.8s
                   all        192        891      0.697      0.737       0.72      0.331
               Missing        104        238      0.505      0.493      0.423       0.12
          Dental Crown         57        172      0.901      0.955      0.952      0.478
            Root Canal         55        165      0.622      0.721      0.657      0.201
                Caries         74        131      0.663      0.618      0.606      0.219
           Broken Down         33         66

In [ ]:
# Append test dataset split to data.yaml if missing
with open("/content/data.yaml", "r+") as f:
    content = f.read()
    if "test:" not in content:
        f.write("test: images/test\n")

metrics_test_v8 = model_v8.val(
    data="/content/data.yaml",
    split="test",
    imgsz=640,
    device=0
)

print("=== YOLOv8m TEST RESULTS ===")
print(f"mAP@0.5      : {metrics_test_v8.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_test_v8.box.map:.4f}")
print(f"Precision    : {metrics_test_v8.box.mp:.4f}")
print(f"Recall       : {metrics_test_v8.box.mr:.4f}")

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,843,813 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 98.2±58.3 MB/s, size: 732.7 KB)
val: Scanning /content/Data_split/labels/test... 198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 198/198 464.5it/s 0.4s
val: New cache created: /content/Data_split/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.9it/s 7.0s
                   all        198       1083       0.68      0.745      0.705      0.325
               Missing        126        321      0.483      0.467      0.411      0.119
          Dental Crown         60        246      0.903      0.951      0.943      0.475
            Root Canal         60        203      0.695      0.739      0.653      0.205
                Caries         90        151      0.626      0.531      0.533  

In [ ]:
import shutil

zip_path = "/content/runs.zip"

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/content",
    base_dir="runs"
)

print("✅ runs folder zipped at:", zip_path)


✅ runs folder zipped at: /content/runs.zip
